# 001 ベースライン実験

## 目的

Breast Cancer Wisconsin データセットを使い、最初の分類ベースラインを作成する。

このNotebookでは、モデル精度を高めることよりも、以下を優先する。

- データセットの構造を確認する
- train / test split を行う
- 最小限の分類モデルを訓練する
- 評価指標を出力する
- 結果を自分の言葉で観察する

## このNotebookの読み方

Notebookは、上から順番に実行する実験ノートである。

各コードセルは、次のどれかを行っている。

1. ライブラリを読み込む
2. データを読み込む
3. データの中身を確認する
4. 訓練用データとテスト用データに分ける
5. モデルを学習する
6. 予測する
7. 評価指標を見る
8. 観察を書く

最初は、各セルを `Shift + Enter` で1つずつ実行する。


## 実験メモ

### データセットを選んだ理由

- scikit-learn に内蔵されており、外部データ取得が不要で再現しやすい
- 二値分類問題であり、初回の評価指標設計に集中しやすい
- 特徴量が数値で揃っており、前処理よりも評価とベースライン構築に集中できる

### 今回の完了条件

- データを読み込める
- 特徴量数、サンプル数、目的変数の分布を確認できる
- Logistic Regression でベースラインを作る
- accuracy, precision, recall, F1 を出力する
- confusion matrix を読み取る
- 観察を日本語で書く


## 1. ライブラリを読み込む

ここでは、今回使う道具を読み込む。

- `numpy`: 数値計算
- `pandas`: 表形式データの操作
- `sklearn.datasets`: 練習用データセット
- `train_test_split`: データ分割
- `LogisticRegression`: 最初の分類モデル
- `metrics`: モデル評価


In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)


## 2. データを読み込む

`load_breast_cancer()` は、乳がん診断に関する二値分類データセットを読み込む。

このデータセットでは、目的変数 `target` は次の意味を持つ。

- `0`: malignant（悪性）
- `1`: benign（良性）

ここで注意すること:

機械学習の分類問題では、単に「正解率が高い」だけでは不十分である。
特にこのデータセットでは、**悪性を良性と誤判定する見逃し**が重要な誤りになる。


In [ ]:
dataset = load_breast_cancer()

X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name="target")

print("特徴量 X の形状:", X.shape)
print("目的変数 y の形状:", y.shape)
print("目的変数のクラス名:", dataset.target_names)
print()
print("目的変数の意味:")
for class_id, class_name in enumerate(dataset.target_names):
    print(f"  {class_id}: {class_name}")
print()
print("目的変数の分布:")
print(y.value_counts().sort_index())


### この出力の読み方

- `特徴量 X の形状` は `(サンプル数, 特徴量数)` を表す
- `目的変数 y の形状` は、正解ラベルの数を表す
- `目的変数の分布` は、悪性と良性がそれぞれ何件あるかを表す

この時点で見るべきこと:

- サンプル数はいくつか
- 特徴量はいくつあるか
- 悪性と良性の件数に偏りがあるか


## 3. 特徴量の先頭を見る

`X.head()` は、特徴量テーブルの先頭5行を表示する。

ここでは、各行が1つのサンプル、各列が1つの特徴量であることを確認する。


In [ ]:
X.head()


## 4. 訓練用データとテスト用データに分ける

モデルの性能を見るには、学習に使ったデータだけで評価してはいけない。

そのため、データを次の2つに分ける。

- 訓練用データ: モデルに学習させる
- テスト用データ: 学習後に性能を確認する

`stratify=y` は、悪性と良性の比率が訓練用・テスト用で大きく崩れないようにする指定である。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("訓練用データ:", X_train.shape, y_train.shape)
print("テスト用データ:", X_test.shape, y_test.shape)
print()
print("訓練用データの目的変数分布:")
print(y_train.value_counts().sort_index())
print()
print("テスト用データの目的変数分布:")
print(y_test.value_counts().sort_index())


## 5. Logistic Regressionで学習する

ここで初めてモデルを作る。

Logistic Regression は、入力された特徴量から、各サンプルが `0: malignant` か `1: benign` かを予測する分類モデルである。

今回の目的は高精度化ではなく、まず「ベースライン」を作ること。
ベースラインとは、今後の改善実験の比較対象になる最初の基準である。


In [ ]:
model = LogisticRegression(max_iter=10_000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("予測が完了しました")
print("予測ラベルの分布:")
print(pd.Series(y_pred, name="pred").value_counts().sort_index())


## 6. 評価指標を見る

ここが観察結果を書くための中心である。

今回見る指標:

- `accuracy`: 全体の正解率
- `precision`: そのクラスだと予測したもののうち、実際に正しかった割合
- `recall`: 実際にそのクラスだったものを、どれだけ拾えたか
- `F1`: precision と recall のバランス

重要:

scikit-learn の `precision_score`, `recall_score`, `f1_score` は、指定しない場合 `1` を陽性クラスとして扱う。
このデータセットでは `1 = benign（良性）` である。
しかし医療的に重要なのは、多くの場合 `0 = malignant（悪性）` を見逃さないことである。

そのため、このNotebookでは悪性と良性の両方について指標を出す。


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

metrics = pd.DataFrame(
    {
        "score": [
            accuracy,
            precision_score(y_test, y_pred, pos_label=0),
            recall_score(y_test, y_pred, pos_label=0),
            f1_score(y_test, y_pred, pos_label=0),
            precision_score(y_test, y_pred, pos_label=1),
            recall_score(y_test, y_pred, pos_label=1),
            f1_score(y_test, y_pred, pos_label=1),
        ]
    },
    index=[
        "accuracy（全体の正解率）",
        "precision: malignant（悪性と予測したものの正確さ）",
        "recall: malignant（実際の悪性を拾えた割合）",
        "F1: malignant（悪性のprecision/recallのバランス）",
        "precision: benign（良性と予測したものの正確さ）",
        "recall: benign（実際の良性を拾えた割合）",
        "F1: benign（良性のprecision/recallのバランス）",
    ],
)

metrics


### 観察欄との対応

Notebook末尾の観察欄には、上の表から次の値を書く。

- `accuracy`: `accuracy（全体の正解率）`
- `precision`: `precision: malignant（悪性と予測したものの正確さ）`
- `recall`: `recall: malignant（実際の悪性を拾えた割合）`
- `F1`: `F1: malignant（悪性のprecision/recallのバランス）`

今回は、悪性検出を重視して `malignant` 側の precision / recall / F1 を記録する。


## 7. confusion matrixを見る

confusion matrix は、予測がどのように当たったか・外れたかを表す表である。

このNotebookでは、行が「実際のラベル」、列が「予測ラベル」である。

|  | 予測: 悪性 | 予測: 良性 |
|---|---:|---:|
| 実際: 悪性 | 悪性を正しく悪性と予測 | 悪性を良性と誤判定 |
| 実際: 良性 | 良性を悪性と誤判定 | 良性を正しく良性と予測 |

特に見るべきなのは、**実際は悪性なのに、良性と予測した数**である。
これは悪性の見逃しであり、医療的には重要な誤りになりやすい。


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["実際: malignant（悪性）", "実際: benign（良性）"],
    columns=["予測: malignant（悪性）", "予測: benign（良性）"],
)

cm_df


In [ ]:
true_malignant_pred_malignant = cm_df.loc["実際: malignant（悪性）", "予測: malignant（悪性）"]
false_negative_malignant = cm_df.loc["実際: malignant（悪性）", "予測: benign（良性）"]
false_positive_malignant = cm_df.loc["実際: benign（良性）", "予測: malignant（悪性）"]
true_benign_pred_benign = cm_df.loc["実際: benign（良性）", "予測: benign（良性）"]

print("confusion matrix の読み取り:")
print(f"- 実際に悪性で、悪性と予測できた数: {true_malignant_pred_malignant}")
print(f"- 実際に悪性だが、良性と誤判定した数（見逃し）: {false_negative_malignant}")
print(f"- 実際に良性だが、悪性と誤判定した数（過検出）: {false_positive_malignant}")
print(f"- 実際に良性で、良性と予測できた数: {true_benign_pred_benign}")


## 8. classification reportを見る

`classification_report` は、クラスごとの precision / recall / F1 をまとめて出す。

ここでは、上で計算した評価指標と同じ内容が、scikit-learn標準形式で表示される。

最初は読み飛ばしてもよい。
今は、`malignant` 行と `benign` 行が分かれていることだけ確認する。


In [ ]:
print(classification_report(y_test, y_pred, target_names=dataset.target_names))


## 9. 観察を書くための下書きを出力する

次のセルは、観察欄に転記するための下書きを出力する。

数値そのものよりも、最後の「confusion matrix から分かること」を自分の言葉で書くことが重要である。


In [ ]:
observation_draft = f"""
- accuracy: {accuracy:.3f}
- precision（悪性）: {precision_score(y_test, y_pred, pos_label=0):.3f}
- recall（悪性）: {recall_score(y_test, y_pred, pos_label=0):.3f}
- F1（悪性）: {f1_score(y_test, y_pred, pos_label=0):.3f}
- confusion matrix から分かること:
  - 実際に悪性だったサンプルのうち、{false_negative_malignant}件を良性と誤判定した。
  - 実際に良性だったサンプルのうち、{false_positive_malignant}件を悪性と誤判定した。
  - このデータセットでは、悪性の見逃しを減らすことが特に重要だと考える。
"""

print(observation_draft)


## 観察

実行後に記入する。

- accuracy:
- precision（悪性）:
- recall（悪性）:
- F1（悪性）:
- confusion matrix から分かること:

## 次の改善仮説

実行後に記入する。

- 
